# 04 — Chiffrement symétrique avec Fernet

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- distinguer chiffrement symétrique et asymétrique ;
- utiliser `Fernet` de la bibliothèque `cryptography` pour chiffrer et déchiffrer ;
- gérer les clés de chiffrement de manière sécurisée ;
- implémenter la rotation de clés avec `MultiFernet` ;
- dériver une clé depuis un mot de passe avec PBKDF2 ;
- identifier les erreurs courantes en cryptographie.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- le hachage (`hashlib`, `hmac`) — notebook 01 ;
- le hachage de mots de passe (`argon2`) — notebook 02 ;
- les tokens et `secrets` — notebook 03 ;
- les types `bytes` et `str`, l'encodage base64.

## Plan

1. Chiffrement symétrique vs asymétrique
2. La bibliothèque `cryptography`
3. Fernet — chiffrement symétrique simple et sûr
4. Chiffrer et déchiffrer
5. Tokens avec TTL (expiration)
6. Dériver une clé depuis un mot de passe
7. Rotation de clés avec `MultiFernet`
8. Chiffrer des fichiers
9. Anti-patterns cryptographiques
10. Synthèse
11. Exercices
12. Ressources

---

## 1. Chiffrement symétrique vs asymétrique

| Critère | Symétrique | Asymétrique |
|---|---|---|
| Clés | 1 clé partagée | 2 clés (publique + privée) |
| Vitesse | Rapide | 100-1000x plus lent |
| Usage | Données au repos, tunnels | Échange de clés, signatures |
| Exemples | AES, ChaCha20, **Fernet** | RSA, Ed25519, ECDH |

En pratique, on utilise souvent les deux : asymétrique pour **échanger la clé**, symétrique pour **chiffrer les données**.

---

## 2. La bibliothèque `cryptography`

`cryptography` est la bibliothèque de référence en Python pour la cryptographie. Elle fournit :
- **Recettes de haut niveau** : `Fernet` (symétrique), signatures, certificats X.509 ;
- **Primitives de bas niveau** : AES, RSA, courbes elliptiques, etc.

> **Installation :** `pip install cryptography`

In [ ]:
try:
    import cryptography
    print(f"cryptography version : {cryptography.__version__}")
except ImportError:
    print("cryptography non installé — pip install cryptography")

---

## 3. Fernet — chiffrement symétrique simple et sûr

Fernet est un **standard de chiffrement authentifié** défini par la bibliothèque `cryptography`. Il combine :
- **AES-128-CBC** pour le chiffrement ;
- **HMAC-SHA256** pour l'authentification (intégrité) ;
- **IV aléatoire** pour chaque message ;
- **Timestamp** intégré pour la gestion de l'expiration.

Fernet garantit que les données chiffrées ne peuvent être ni lues ni modifiées sans la clé.

### Pourquoi Fernet et pas AES « à la main » ?

| « AES à la main » | Fernet |
|---|---|
| Choisir le mode (CBC, CTR, GCM...) | Fait pour vous |
| Gérer l'IV/nonce | Automatique |
| Gérer le padding | Automatique |
| Ajouter HMAC pour l'intégrité | Intégré |
| Risque d'erreur cryptographique | Minimisé |

**Si vous n'êtes pas cryptographe, utilisez Fernet.** C'est conçu pour être impossible à mal utiliser.

---

## 4. Chiffrer et déchiffrer

In [ ]:
try:
    from cryptography.fernet import Fernet

    # Générer une clé
    cle = Fernet.generate_key()
    print(f"Clé : {cle.decode()}")
    print(f"Type : {type(cle)}, longueur : {len(cle)} octets")
except ImportError:
    print("cryptography non installé")

La clé Fernet est une chaîne base64 de 32 octets (256 bits) : 16 octets pour AES-128 + 16 octets pour HMAC-SHA256.

In [ ]:
try:
    from cryptography.fernet import Fernet

    cle = Fernet.generate_key()
    f = Fernet(cle)

    # Chiffrer
    message = b"Donnees sensibles : carte 4242-4242-4242-4242"
    token = f.encrypt(message)
    print(f"Chiffré : {token[:60]}...")
    print(f"Longueur : {len(token)} octets")

    # Déchiffrer
    clair = f.decrypt(token)
    print(f"Déchiffré : {clair.decode()}")

except ImportError:
    print("cryptography non installé")

### Chiffrer du texte (str → bytes → chiffré)

In [ ]:
try:
    from cryptography.fernet import Fernet

    cle = Fernet.generate_key()
    f = Fernet(cle)

    texte = "Message en français avec des accents : éàü"
    token = f.encrypt(texte.encode("utf-8"))
    clair = f.decrypt(token).decode("utf-8")
    print(f"Original  : {texte}")
    print(f"Déchiffré : {clair}")

except ImportError:
    print("cryptography non installé")

### Chaque chiffrement produit un résultat différent

In [ ]:
try:
    from cryptography.fernet import Fernet

    cle = Fernet.generate_key()
    f = Fernet(cle)
    msg = b"hello"

    t1 = f.encrypt(msg)
    t2 = f.encrypt(msg)
    print(f"Token 1 : {t1[:40]}...")
    print(f"Token 2 : {t2[:40]}...")
    print(f"Identiques : {t1 == t2}")  # False !
    print(f"Déchiffrés identiques : {f.decrypt(t1) == f.decrypt(t2)}")  # True

except ImportError:
    print("cryptography non installé")

Chaque appel à `encrypt()` utilise un **IV aléatoire** différent, ce qui rend deux chiffrements du même message indistinguables. C'est une propriété de sécurité importante (**IND-CPA**).

### Mauvaise clé → erreur

In [ ]:
try:
    from cryptography.fernet import Fernet, InvalidToken

    cle1 = Fernet.generate_key()
    cle2 = Fernet.generate_key()

    f1 = Fernet(cle1)
    f2 = Fernet(cle2)

    token = f1.encrypt(b"secret")

    try:
        f2.decrypt(token)
    except InvalidToken:
        print("InvalidToken : la clé ne correspond pas (ou le token est altéré)")

except ImportError:
    print("cryptography non installé")

---

## 5. Tokens avec TTL (expiration)

Fernet intègre un **timestamp** dans chaque token. La méthode `decrypt(token, ttl=...)` refuse les tokens plus vieux que `ttl` secondes.

In [ ]:
try:
    import time
    from cryptography.fernet import Fernet, InvalidToken

    cle = Fernet.generate_key()
    f = Fernet(cle)

    token = f.encrypt(b"donnees temporaires")

    # Déchiffrer avec un TTL de 10 secondes
    clair = f.decrypt(token, ttl=10)
    print(f"OK (dans le TTL) : {clair}")

    # Attendre et réessayer
    time.sleep(2)
    try:
        f.decrypt(token, ttl=1)
    except InvalidToken:
        print("Token expiré (TTL dépassé)")

except ImportError:
    print("cryptography non installé")

### Extraire le timestamp

In [ ]:
try:
    from cryptography.fernet import Fernet
    from datetime import datetime

    cle = Fernet.generate_key()
    f = Fernet(cle)
    token = f.encrypt(b"test")

    ts = f.extract_timestamp(token)
    print(f"Timestamp : {ts}")
    print(f"Date : {datetime.fromtimestamp(ts)}")

except ImportError:
    print("cryptography non installé")

---

## 6. Dériver une clé depuis un mot de passe

Si vous n'avez pas de clé aléatoire mais un **mot de passe**, il faut le transformer en clé avec une fonction de dérivation (KDF).

In [ ]:
try:
    import base64
    import os
    from cryptography.fernet import Fernet
    from cryptography.hazmat.primitives import hashes
    from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

    mot_de_passe = b"MonMotDePasseUtilisateur"
    sel = os.urandom(16)

    # Dériver une clé de 32 octets
    kdf = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=32,
        salt=sel,
        iterations=600_000,
    )
    cle = base64.urlsafe_b64encode(kdf.derive(mot_de_passe))

    # Utiliser avec Fernet
    f = Fernet(cle)
    token = f.encrypt(b"Données protégées par mot de passe")
    print(f"Chiffré : {token[:50]}...")

    # Pour déchiffrer, il faut le même mot de passe ET le même sel
    kdf2 = PBKDF2HMAC(
        algorithm=hashes.SHA256(),
        length=32,
        salt=sel,  # même sel !
        iterations=600_000,
    )
    cle2 = base64.urlsafe_b64encode(kdf2.derive(mot_de_passe))
    f2 = Fernet(cle2)
    print(f"Déchiffré : {f2.decrypt(token).decode()}")

except ImportError:
    print("cryptography non installé")

**Le sel doit être stocké à côté du message chiffré** (il n'est pas secret). Sans le sel, impossible de recalculer la clé.

---

## 7. Rotation de clés avec `MultiFernet`

`MultiFernet` permet de **migrer progressivement** d'une ancienne clé vers une nouvelle, sans tout re-chiffrer d'un coup.

In [ ]:
try:
    from cryptography.fernet import Fernet, MultiFernet

    ancienne_cle = Fernet.generate_key()
    nouvelle_cle = Fernet.generate_key()

    f_ancienne = Fernet(ancienne_cle)
    f_nouvelle = Fernet(nouvelle_cle)

    # MultiFernet essaie les clés dans l'ordre
    multi = MultiFernet([f_nouvelle, f_ancienne])

    # Chiffrer avec l'ancienne clé
    token_ancien = f_ancienne.encrypt(b"données anciennes")

    # MultiFernet peut déchiffrer avec l'ancienne clé
    clair = multi.decrypt(token_ancien)
    print(f"Déchiffré : {clair.decode()}")

    # rotate() re-chiffre avec la première clé (la nouvelle)
    token_migre = multi.rotate(token_ancien)
    print(f"Token migré : {token_migre[:40]}...")

    # Maintenant seule la nouvelle clé suffit
    clair2 = f_nouvelle.decrypt(token_migre)
    print(f"Avec nouvelle clé : {clair2.decode()}")

except ImportError:
    print("cryptography non installé")

### Stratégie de rotation

1. Générer une nouvelle clé ;
2. Configurer `MultiFernet([nouvelle, ancienne])` ;
3. Les nouvelles données sont chiffrées avec la nouvelle clé ;
4. Progressivement, migrer les anciennes données avec `rotate()` ;
5. Quand tout est migré, retirer l'ancienne clé.

---

## 8. Chiffrer des fichiers

In [ ]:
try:
    import os
    import tempfile
    from cryptography.fernet import Fernet

    def chiffrer_fichier(chemin_source: str, chemin_dest: str, cle: bytes) -> None:
        f = Fernet(cle)
        with open(chemin_source, "rb") as src:
            donnees = src.read()
        token = f.encrypt(donnees)
        with open(chemin_dest, "wb") as dst:
            dst.write(token)

    def dechiffrer_fichier(chemin_source: str, chemin_dest: str, cle: bytes) -> None:
        f = Fernet(cle)
        with open(chemin_source, "rb") as src:
            token = src.read()
        donnees = f.decrypt(token)
        with open(chemin_dest, "wb") as dst:
            dst.write(donnees)

    # Test
    cle = Fernet.generate_key()
    with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
        f.write("Données confidentielles du projet X\n" * 100)
        fichier_clair = f.name

    fichier_chiffre = fichier_clair + ".enc"
    fichier_dechiffre = fichier_clair + ".dec"

    chiffrer_fichier(fichier_clair, fichier_chiffre, cle)
    dechiffrer_fichier(fichier_chiffre, fichier_dechiffre, cle)

    with open(fichier_clair) as f1, open(fichier_dechiffre) as f2:
        print(f"Identiques : {f1.read() == f2.read()}")

    for p in [fichier_clair, fichier_chiffre, fichier_dechiffre]:
        os.unlink(p)

except ImportError:
    print("cryptography non installé")

**Attention :** Fernet charge tout le fichier en mémoire. Pour les fichiers de plus de quelques centaines de Mo, utilisez `ChaCha20Poly1305` ou `AESGCM` en mode streaming.

---

## 9. Anti-patterns cryptographiques

### Ne JAMAIS implémenter votre propre crypto

```python
# DANGEREUX — ne faites jamais cela !
def mon_chiffrement(msg, cle):
    return bytes(b ^ cle for b in msg)
```

Le XOR simple est trivial à casser. Utilisez `Fernet` ou `AESGCM`.

### Ne pas réutiliser un IV/nonce

Fernet le fait automatiquement, mais si vous utilisez des primitives bas-niveau :

```python
# DANGEREUX — IV réutilisé
cipher = AES.new(key, AES.MODE_CBC, iv=FIXED_IV)  # JAMAIS !
```

### Ne pas stocker la clé avec les données chiffrées

| Stockage | Correct ? |
|---|---|
| Clé dans le même fichier que les données | Non |
| Clé dans le code source (git) | Non |
| Clé en variable d'environnement | Acceptable |
| Clé dans un vault (AWS KMS, HashiCorp) | Oui |

### Chiffrer sans authentifier

Le chiffrement seul ne protège pas contre la **modification**. Fernet intègre HMAC-SHA256, mais certaines primitives (AES-CBC seul) ne le font pas.

**Toujours utiliser du chiffrement authentifié** : Fernet, AES-GCM, ChaCha20-Poly1305.

---

## 10. Synthèse

| Outil | Usage |
|---|---|
| `Fernet.generate_key()` | Générer une clé aléatoire |
| `Fernet(key).encrypt(data)` | Chiffrer |
| `Fernet(key).decrypt(token)` | Déchiffrer |
| `Fernet(key).decrypt(token, ttl=N)` | Déchiffrer avec expiration |
| `MultiFernet([new, old])` | Rotation de clés |
| `PBKDF2HMAC` | Dériver une clé depuis un mot de passe |

**Règles à retenir :**
- **Fernet** pour le chiffrement symétrique : simple, sûr, difficile à mal utiliser.
- Chaque `encrypt()` produit un résultat différent (IV aléatoire).
- Stockez les clés **séparément** des données chiffrées.
- Utilisez `PBKDF2HMAC` (600K+ itérations) pour dériver une clé depuis un mot de passe.
- `MultiFernet` pour la rotation de clés sans downtime.
- **Jamais** de crypto maison, **jamais** d'IV fixe, **toujours** du chiffrement authentifié.

---

## 11. Exercices

### Exercice 1 — Chiffrer/déchiffrer un message *(facile)*

Générer une clé Fernet, chiffrer le message `"Python est génial !"`, puis le déchiffrer et vérifier que le résultat est identique.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Chiffrement_fernet", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from cryptography.fernet import Fernet

cle = Fernet.generate_key()
f = Fernet(cle)

message = "Python est génial !"
token = f.encrypt(message.encode("utf-8"))
clair = f.decrypt(token).decode("utf-8")

print(f"Original  : {message}")
print(f"Chiffré   : {token[:40]}...")
print(f"Déchiffré : {clair}")
assert message == clair
```

</details>

### Exercice 2 — Coffre-fort avec mot de passe *(moyen)*

Implémenter une classe `CoffreFort` qui :
1. Prend un mot de passe à la construction ;
2. Dérive une clé avec PBKDF2 (sel aléatoire) ;
3. Offre `stocker(nom, donnees)` et `recuperer(nom)` ;
4. Stocke tout en mémoire (dict chiffré).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Chiffrement_fernet", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import base64
import os
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

class CoffreFort:
    def __init__(self, mot_de_passe: str):
        self._sel = os.urandom(16)
        kdf = PBKDF2HMAC(
            algorithm=hashes.SHA256(),
            length=32,
            salt=self._sel,
            iterations=600_000,
        )
        cle = base64.urlsafe_b64encode(kdf.derive(mot_de_passe.encode()))
        self._fernet = Fernet(cle)
        self._coffre = {}

    def stocker(self, nom: str, donnees: str) -> None:
        self._coffre[nom] = self._fernet.encrypt(donnees.encode("utf-8"))

    def recuperer(self, nom: str) -> str | None:
        token = self._coffre.get(nom)
        if token is None:
            return None
        return self._fernet.decrypt(token).decode("utf-8")

    def lister(self) -> list[str]:
        return list(self._coffre.keys())

coffre = CoffreFort("MotDePasse123!")
coffre.stocker("api_key", "sk_live_abc123")
coffre.stocker("db_password", "p@ssw0rd!")

print(f"Noms : {coffre.lister()}")
print(f"api_key : {coffre.recuperer('api_key')}")
print(f"db_password : {coffre.recuperer('db_password')}")
```

</details>

### Exercice 3 — Rotation de clés automatique *(moyen)*

Implémenter un système qui :
1. Maintient un historique de clés (max 3) ;
2. Chiffre toujours avec la clé la plus récente ;
3. Peut déchiffrer avec n'importe quelle clé de l'historique ;
4. Offre une méthode `rotation()` qui génère une nouvelle clé.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Chiffrement_fernet", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from cryptography.fernet import Fernet, MultiFernet

class KeyManager:
    def __init__(self, max_cles: int = 3):
        self.max_cles = max_cles
        self._cles = [Fernet.generate_key()]

    def _multi(self) -> MultiFernet:
        return MultiFernet([Fernet(k) for k in self._cles])

    def rotation(self) -> None:
        nouvelle = Fernet.generate_key()
        self._cles.insert(0, nouvelle)
        if len(self._cles) > self.max_cles:
            self._cles.pop()
        print(f"Rotation : {len(self._cles)} clés actives")

    def chiffrer(self, data: bytes) -> bytes:
        return self._multi().encrypt(data)

    def dechiffrer(self, token: bytes) -> bytes:
        return self._multi().decrypt(token)

    def migrer(self, token: bytes) -> bytes:
        return self._multi().rotate(token)

km = KeyManager()
t1 = km.chiffrer(b"données v1")
km.rotation()
t2 = km.chiffrer(b"données v2")

print(f"v1 : {km.dechiffrer(t1)}")
print(f"v2 : {km.dechiffrer(t2)}")

# Migrer l'ancien token
t1_migre = km.migrer(t1)
print(f"v1 migrée : {km.dechiffrer(t1_migre)}")
```

</details>

### Exercice 4 — Chiffrement de fichiers par blocs *(difficile)*

Fernet charge tout en mémoire. Implémenter un chiffrement de fichiers **par blocs** en utilisant `cryptography.hazmat.primitives.ciphers` (AES-GCM) pour supporter les gros fichiers.

Interface :
```python
chiffrer_gros_fichier(chemin_src, chemin_dst, cle)
dechiffrer_gros_fichier(chemin_src, chemin_dst, cle)
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="04_Chiffrement_fernet", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import os
import struct
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

CHUNK_SIZE = 64 * 1024  # 64 Ko par bloc

def chiffrer_gros_fichier(chemin_src, chemin_dst, cle):
    aesgcm = AESGCM(cle)
    with open(chemin_src, "rb") as src, open(chemin_dst, "wb") as dst:
        while True:
            bloc = src.read(CHUNK_SIZE)
            if not bloc:
                break
            nonce = os.urandom(12)
            chiffre = aesgcm.encrypt(nonce, bloc, None)
            # Écrire : taille du bloc chiffré + nonce + bloc chiffré
            dst.write(struct.pack(">I", len(chiffre)))
            dst.write(nonce)
            dst.write(chiffre)

def dechiffrer_gros_fichier(chemin_src, chemin_dst, cle):
    aesgcm = AESGCM(cle)
    with open(chemin_src, "rb") as src, open(chemin_dst, "wb") as dst:
        while True:
            header = src.read(4)
            if not header:
                break
            taille = struct.unpack(">I", header)[0]
            nonce = src.read(12)
            chiffre = src.read(taille)
            clair = aesgcm.decrypt(nonce, chiffre, None)
            dst.write(clair)

# Test
import tempfile
cle = AESGCM.generate_key(bit_length=256)

with tempfile.NamedTemporaryFile(delete=False) as f:
    f.write(b"x" * 200_000)
    src = f.name

enc = src + ".enc"
dec = src + ".dec"

chiffrer_gros_fichier(src, enc, cle)
dechiffrer_gros_fichier(enc, dec, cle)

with open(src, "rb") as f1, open(dec, "rb") as f2:
    print(f"Identiques : {f1.read() == f2.read()}")

for p in [src, enc, dec]:
    os.unlink(p)
```

</details>

---

## 12. Ressources

- [`cryptography` — documentation Fernet](https://cryptography.io/en/latest/fernet/)
- [`cryptography` — documentation AESGCM](https://cryptography.io/en/latest/hazmat/primitives/aead/)
- [Fernet Spec](https://github.com/fernet/spec/blob/master/Spec.md)
- [OWASP — Cryptographic Storage Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/Cryptographic_Storage_Cheat_Sheet.html)